In [6]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import time
import random
import pandas as pd
import json
import os

In [3]:
# ==========================================
# 1. CẤU HÌNH VƯỢT TƯỜNG LỬA
# ==========================================
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

# CHỈ CÀO CÁC DANH MỤC TẠI ĐÀ NẴNG
DANH_MUC = [
    {"Loại_Phòng": "Phòng Trọ", "URL": "https://phongtro123.com/tinh-thanh/da-nang"},
    {"Loại_Phòng": "Nhà Nguyên Căn", "URL": "https://phongtro123.com/cho-thue-nha-nguyen-can-da-nang"},
    {"Loại_Phòng": "Chung Cư", "URL": "https://phongtro123.com/cho-thue-can-ho-chung-cu-da-nang"},
    {"Loại_Phòng": "Chung Cư Mini", "URL": "https://phongtro123.com/cho-thue-can-ho-chung-cu-mini-da-nang"}
]

MAX_PAGES = 999 # Cào toàn bộ đến khi hết dữ liệu
all_rooms = []
seen_urls = set() # Chống bẫy lặp trang

# ==========================================
# 2. HÀM FEATURE ENGINEERING
# ==========================================
def extract_utilities(text_content):
    d = str(text_content).lower()
    return {
        "Máy_Lạnh": 1 if re.search(r'(máy lạnh|điều hòa|điều hoà)', d) else 0,
        "Nóng_Lạnh": 1 if re.search(r'(nóng lạnh|máy nước nóng|bình nước nóng)', d) else 0,
        "Thang_Máy": 1 if re.search(r'(thang máy)', d) else 0,
        "Tủ_Lạnh": 1 if re.search(r'(tủ lạnh)', d) else 0,
        "Máy_Giặt": 1 if re.search(r'(máy giặt)', d) else 0,
        "Đầy_Đủ_Nội_Thất": 1 if re.search(r'(đầy đủ nội thất|full nội thất|nội thất đầy đủ)', d) else 0,
        "Có_Gác_Lửng": 1 if re.search(r'(có gác|gác lửng|gác xép)', d) else 0,
        "Bảo_Vệ_An_Ninh": 1 if re.search(r'(bảo vệ|an ninh|camera)', d) else 0,
        "Giờ_Tự_Do": 1 if re.search(r'(giờ giấc tự do|giờ tự do|chìa khóa trao tay)', d) else 0,
        "Không_Chung_Chủ": 1 if re.search(r'(không chung chủ|lối đi riêng)', d) else 0
    }

def extract_price(price_str):
    if not price_str: return None
    p = str(price_str).lower().replace(",", ".")
    match = re.search(r'(\d+(?:\.\d+)?)', p)
    if not match: return None
    val = float(match.group(1))
    
    if 'triệu' in p: return val
    if 'đồng' in p or 'vnđ' in p: return val / 1000000
    if 'k' in p: return val / 1000
    return val

# ==========================================
# 3. QUÁ TRÌNH CÀO
# ==========================================
print("🚀 BẮT ĐẦU CÀO DỮ LIỆU PHÒNG TRỌ ĐÀ NẴNG...")
print("⚠️ LƯU Ý: Bạn có thể nhấn Ctrl + C bất cứ lúc nào để DỪNG và LƯU dữ liệu.")

try:
    for dm in DANH_MUC:
        loai_phong = dm["Loại_Phòng"]
        print(f"\n👉 ĐANG QUÉT DANH MỤC: {loai_phong.upper()}")
        
        for page in range(1, MAX_PAGES + 1):
            list_url = f"{dm['URL']}?page={page}" if page > 1 else dm['URL']
            print(f"  + Trang {page}...")
            
            try:
                res = scraper.get(list_url, timeout=15)
                if res.status_code != 200: continue
                
                soup = BeautifulSoup(res.text, "html.parser")
                scripts = soup.find_all('script', type='application/ld+json')
                
                json_rooms = []
                has_new_room = False 
                
                for script in scripts:
                    try:
                        raw_json = script.string.strip() if script.string else ""
                        if not raw_json: continue
                        json_data = json.loads(raw_json)
                        
                        if isinstance(json_data, dict) and json_data.get('@type') in ['Hostel', 'Apartment', 'SingleFamilyResidence']:
                            url_val = json_data.get('url', '')
                            if not url_val: continue
                            
                            if url_val not in seen_urls:
                                seen_urls.add(url_val)
                                has_new_room = True 
                                
                                room = {
                                    "URL": url_val,
                                    "Loại_Phòng": loai_phong,
                                    "Tiêu_Đề": json_data.get('name'),
                                    "Điện_Thoại": json_data.get('telephone'),
                                    "Mô_Tả_Tạm": json_data.get('description', '') # Lưu tạm mô tả để quét tiện ích
                                }
                                
                                code_match = re.search(r'-pr(\d+)\.html', url_val)
                                room["Mã_Tin"] = code_match.group(1) if code_match else None
                                
                                if 'floorSize' in json_data and isinstance(json_data['floorSize'], dict):
                                    room["Diện_Tích_m2"] = float(json_data['floorSize'].get('value', 0))
                                    
                                if 'priceRange' in json_data:
                                    room["Giá_Cho_Thuê"] = int(json_data.get('priceRange', 0)) / 1000000
                                    
                                address_obj = json_data.get('address', {})
                                if isinstance(address_obj, dict):
                                    full_address = address_obj.get('streetAddress', '')
                                    room["Địa_Chỉ_Chi_Tiết"] = full_address
                                    qh_match = re.search(r'(?:Quận|Huyện)\s+([^,]+)', full_address, re.IGNORECASE)
                                    room["Quận_Huyện"] = qh_match.group(1).strip() if qh_match else "Không rõ"
                                        
                                json_rooms.append(room)
                    except Exception:
                        continue
                
                if not has_new_room:
                    print("    -> [!] Đã hết tin mới ở danh mục này. Tự động chuyển danh mục!")
                    break
                    
                print(f"    -> Đã lấy sơ bộ {len(json_rooms)} phòng mới. Đang vào chi tiết...")
                
                found_rooms = 0
                for room in json_rooms:
                    try:
                        detail_res = scraper.get(room["URL"], timeout=15)
                        detail_soup = BeautifulSoup(detail_res.text, "html.parser")
                        
                        # 1. Lấy Giá
                        if "Giá_Cho_Thuê" not in room or not room["Giá_Cho_Thuê"]:
                            price_tag = detail_soup.select_one(".post-price, .item-price, .text-green")
                            if price_tag:
                                room["Giá_Cho_Thuê"] = extract_price(price_tag.text)
                                
                        # 2. Lấy Diện tích
                        if "Diện_Tích_m2" not in room or not room["Diện_Tích_m2"]:
                            for span in detail_soup.find_all('span'):
                                span_text = span.text.strip().lower()
                                if 'm2' in span_text or 'm²' in span_text:
                                    a_match = re.search(r'(\d+(?:\.\d+)?)\s*m', span_text)
                                    if a_match:
                                        room["Diện_Tích_m2"] = float(a_match.group(1))
                                        break 
                        
                        # 3. Lấy Thời gian đăng (Lấy từ thẻ time)
                        time_tag = detail_soup.find('time')
                        if time_tag and time_tag.has_attr('title'):
                            # Cắt bỏ chữ "Cập nhật: " nếu có
                            room["Thời_Gian_Đăng"] = time_tag['title'].replace("Cập nhật:", "").strip()
                        else:
                            room["Thời_Gian_Đăng"] = "Không rõ"
                        
                        # 4. Quét tiện ích
                        desc_tag = detail_soup.select_one(".post-content, .section-content")
                        if desc_tag:
                            room["Mô_Tả_Tạm"] = desc_tag.text.strip().replace('\n', ' ')
                        
                        noi_bat_text = ""
                        noi_bat_items = detail_soup.select(".col-3 .text-body")
                        for item in noi_bat_items:
                            if 'opacity-75' not in item.get('class', []) and item.text:
                                noi_bat_text += item.text.strip() + " | "
                                
                        full_text = str(room.get("Tiêu_Đề","")) + " " + str(room.get("Mô_Tả_Tạm","")) + " " + noi_bat_text
                        room.update(extract_utilities(full_text))
                        
                        all_rooms.append(room)
                        found_rooms += 1
                        
                        time.sleep(random.uniform(0.5, 1.5))
                        
                    except Exception as e:
                        continue
                        
                print(f"    -> Đã hoàn tất chi tiết {found_rooms} phòng.")
                
            except Exception as e:
                print(f"  [!] Lỗi kết nối trang: {e}")
                break

except KeyboardInterrupt:
    print("\n[!] ⛔ BẠN VỪA BẤM DỪNG (CTRL+C). Đang tiến hành lưu toàn bộ dữ liệu đã cào được...")

# ==========================================
# 4. XUẤT DỮ LIỆU (KHÔNG CHỨA MÔ TẢ & URL)
# ==========================================
if all_rooms:
    df = pd.DataFrame(all_rooms)
    
    # Danh sách cột CHÍNH THỨC xuất ra file CSV (Đã bỏ Mô_Tả_Tạm và URL, thêm Thời_Gian_Đăng)
    cols = ['Mã_Tin', 'Thời_Gian_Đăng', 'Loại_Phòng', 'Quận_Huyện', 'Giá_Cho_Thuê', 'Diện_Tích_m2', 
            'Đầy_Đủ_Nội_Thất', 'Có_Gác_Lửng', 'Máy_Lạnh', 'Nóng_Lạnh', 'Thang_Máy', 
            'Tủ_Lạnh', 'Máy_Giặt', 'Bảo_Vệ_An_Ninh', 'Giờ_Tự_Do', 'Không_Chung_Chủ', 
            'Tiêu_Đề', 'Điện_Thoại', 'Địa_Chỉ_Chi_Tiết']
            
    # Lọc data theo đúng danh sách cột ở trên
    df = df[[c for c in cols if c in df.columns]]
    
    df = df.dropna(subset=['Giá_Cho_Thuê', 'Diện_Tích_m2'])
    
    df.to_csv('data_phongtro_danang_sach.csv', index=False, encoding='utf-8-sig')
    print(f"\n🎉 XUẤT SẮC! Đã lưu an toàn {len(df)} phòng trọ vào file 'data_phongtro_danang_sach.csv'")
else:
    print("\n[!] Chưa cào được dữ liệu nào.")

🚀 BẮT ĐẦU CÀO DỮ LIỆU PHÒNG TRỌ ĐÀ NẴNG...
⚠️ LƯU Ý: Bạn có thể nhấn Ctrl + C bất cứ lúc nào để DỪNG và LƯU dữ liệu.

👉 ĐANG QUÉT DANH MỤC: PHÒNG TRỌ
  + Trang 1...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 2...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 3...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 4...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 5...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 6...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 7...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 8...
    -> Đã lấy sơ bộ 20 phòng mới. Đang vào chi tiết...
    -> Đã hoàn tất chi t

In [ ]:
# 1. CẤU HÌNH VƯỢT TƯỜNG LỬA
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

DANH_MUC = [
    {"Thành_Phố": "Đà Nẵng", "URL": "https://phongtro123.com/tinh-thanh/da-nang"},
    {"Thành_Phố": "Hồ Chí Minh", "URL": "https://phongtro123.com/tinh-thanh/ho-chi-minh"},
    {"Thành_Phố": "Hà Nội", "URL": "https://phongtro123.com/tinh-thanh/ha-noi"}
]

MAX_PAGES = 50 
FILE_NAME = 'raw_data.csv'

all_rooms = []
seen_urls = set() 
df_old = None

# 1.5. HỆ THỐNG KHÔI PHỤC TIẾN TRÌNH (RESUME)
if os.path.exists(FILE_NAME):
    print(f"[*] Phát hiện file dữ liệu cũ '{FILE_NAME}'. Đang nạp dữ liệu để cào tiếp...")
    try:
        df_old = pd.read_csv(FILE_NAME)
        all_rooms = df_old.to_dict('records')
        # Bổ sung key URL tạm để chống trùng, nếu file CSV cũ không lưu URL
        # Nếu CSV cũ đã xóa URL, đoạn này sẽ chỉ lấy các tiêu đề hoặc ID để set
        print(f"[*] Đã khôi phục thành công {len(all_rooms)} phòng cũ!")
    except Exception as e:
        print(f"[!] Lỗi khi đọc file cũ: {e}")

# 2. HÀM FEATURE ENGINEERING
def extract_utilities(text_content):
    d = str(text_content).lower()
    return {
        "Máy_Lạnh": 1 if re.search(r'(máy lạnh|điều hòa|điều hoà)', d) else 0,
        "Nóng_Lạnh": 1 if re.search(r'(nóng lạnh|máy nước nóng|bình nước nóng)', d) else 0,
        "Thang_Máy": 1 if re.search(r'(thang máy)', d) else 0,
        "Tủ_Lạnh": 1 if re.search(r'(tủ lạnh)', d) else 0,
        "Máy_Giặt": 1 if re.search(r'(máy giặt)', d) else 0,
        "Đầy_Đủ_Nội_Thất": 1 if re.search(r'(đầy đủ nội thất|full nội thất|nội thất đầy đủ)', d) else 0,
        "Có_Gác_Lửng": 1 if re.search(r'(có gác|gác lửng|gác xép)', d) else 0,
        "Bảo_Vệ_An_Ninh": 1 if re.search(r'(bảo vệ|an ninh|camera)', d) else 0,
        "Giờ_Tự_Do": 1 if re.search(r'(giờ giấc tự do|giờ tự do|chìa khóa trao tay)', d) else 0,
        "Không_Chung_Chủ": 1 if re.search(r'(không chung chủ|lối đi riêng)', d) else 0
    }

def extract_price(price_str):
    if not price_str: return None
    p = str(price_str).lower().replace(",", ".")
    match = re.search(r'(\d+(?:\.\d+)?)', p)
    if not match: return None
    val = float(match.group(1))
    
    if 'triệu' in p: return val
    if 'đồng' in p or 'vnđ' in p: return val / 1000000
    if 'k' in p: return val / 1000
    return val

# 3. QUÁ TRÌNH CÀO
try:
    for dm in DANH_MUC:
        thanh_pho = dm["Thành_Phố"]
        
        start_page = 1
        if df_old is not None and thanh_pho in df_old['Thành_Phố'].values:
            so_luong_da_cao = len(df_old[df_old['Thành_Phố'] == thanh_pho])
            start_page = (so_luong_da_cao // 20) + 1 
            print(f"\n👉 ĐANG QUÉT: {thanh_pho.upper()} (Chạy tiếp từ Trang {start_page})")
        else:
            print(f"\n👉 ĐANG QUÉT: {thanh_pho.upper()} (Bắt đầu từ Trang 1)")
            
        if start_page > MAX_PAGES:
            print(f"    -> Đã cào đủ {MAX_PAGES} trang cho {thanh_pho}. Bỏ qua!")
            continue
        
        for page in range(start_page, MAX_PAGES + 1):
            list_url = f"{dm['URL']}?page={page}" if page > 1 else dm['URL']
            print(f"  + Trang {page} / {MAX_PAGES}...")
            
            try:
                res = scraper.get(list_url, timeout=15)
                if res.status_code != 200: continue
                
                soup = BeautifulSoup(res.text, "html.parser")
                scripts = soup.find_all('script', type='application/ld+json')
                
                json_rooms = []
                has_new_room = False 
                
                for script in scripts:
                    try:
                        raw_json = script.string.strip() if script.string else ""
                        if not raw_json: continue
                        json_data = json.loads(raw_json)
                        
                        if isinstance(json_data, dict) and json_data.get('@type') in ['Hostel', 'Apartment']:
                            url_val = json_data.get('url', '')
                            if not url_val: continue
                            
                            # Dùng Mã_Tin (ID) để kiểm tra trùng thay vì URL, vì file CSV cũ đã bị cắt URL
                            code_match = re.search(r'-pr(\d+)\.html', url_val)
                            ma_tin_val = code_match.group(1) if code_match else None
                            
                            if ma_tin_val and ma_tin_val not in seen_urls:
                                seen_urls.add(ma_tin_val)
                                has_new_room = True 
                                
                                room = {
                                    "Mã_Tin": ma_tin_val,
                                    "URL": url_val,
                                    "Thành_Phố": thanh_pho,
                                    "Tiêu_Đề": json_data.get('name'),
                                    "Điện_Thoại": json_data.get('telephone'),
                                    "Mô_Tả_Tạm": json_data.get('description', '') 
                                }
                                
                                if 'floorSize' in json_data and isinstance(json_data['floorSize'], dict):
                                    room["Diện_Tích_m2"] = float(json_data['floorSize'].get('value', 0))
                                    
                                if 'priceRange' in json_data:
                                    room["Giá_Cho_Thuê"] = int(json_data.get('priceRange', 0)) / 1000000
                                    
                                address_obj = json_data.get('address', {})
                                if isinstance(address_obj, dict):
                                    full_address = address_obj.get('streetAddress', '')
                                    room["Địa_Chỉ_Chi_Tiết"] = full_address
                                    qh_match = re.search(r'(?:Quận|Huyện)\s+([^,]+)', full_address, re.IGNORECASE)
                                    room["Quận_Huyện"] = qh_match.group(1).strip() if qh_match else "Không rõ"
                                        
                                json_rooms.append(room)
                    except Exception:
                        continue
                
                if not has_new_room:
                    print(f"    -> [!] Đã quét hết tin mới tại {thanh_pho}. Chuyển thành phố khác!")
                    break
                    
                print(f"    -> Đã lấy sơ bộ {len(json_rooms)} phòng. Đang vào chi tiết...")
                
                found_rooms = 0
                for room in json_rooms:
                    try:
                        detail_res = scraper.get(room["URL"], timeout=15)
                        detail_soup = BeautifulSoup(detail_res.text, "html.parser")
                        
                        if "Giá_Cho_Thuê" not in room or pd.isna(room.get("Giá_Cho_Thuê")):
                            price_tag = detail_soup.select_one(".post-price, .item-price, .text-green")
                            if price_tag:
                                room["Giá_Cho_Thuê"] = extract_price(price_tag.text)
                                
                        if "Diện_Tích_m2" not in room or pd.isna(room.get("Diện_Tích_m2")):
                            for span in detail_soup.find_all('span'):
                                span_text = span.text.strip().lower()
                                if 'm2' in span_text or 'm²' in span_text:
                                    a_match = re.search(r'(\d+(?:\.\d+)?)\s*m', span_text)
                                    if a_match:
                                        room["Diện_Tích_m2"] = float(a_match.group(1))
                                        break 
                        
                        time_tag = detail_soup.find('time')
                        if time_tag and time_tag.has_attr('title'):
                            room["Thời_Gian_Đăng"] = time_tag['title'].replace("Cập nhật:", "").strip()
                        else:
                            room["Thời_Gian_Đăng"] = "Không rõ"
                        
                        desc_tag = detail_soup.select_one(".post-content, .section-content")
                        if desc_tag:
                            room["Mô_Tả_Tạm"] = desc_tag.text.strip().replace('\n', ' ')
                        
                        noi_bat_text = ""
                        noi_bat_items = detail_soup.select(".col-3 .text-body")
                        for item in noi_bat_items:
                            if 'opacity-75' not in item.get('class', []) and item.text:
                                noi_bat_text += item.text.strip() + " | "
                                
                        full_text = str(room.get("Tiêu_Đề","")) + " " + str(room.get("Mô_Tả_Tạm","")) + " " + noi_bat_text
                        
                        room.update(extract_utilities(full_text))
                        
                        all_rooms.append(room)
                        found_rooms += 1
                        
                        time.sleep(random.uniform(0.5, 1.5))
                        
                    except Exception as e:
                        continue
                        
                print(f"    -> Đã hoàn tất chi tiết {found_rooms} phòng.")
                
            except Exception as e:
                print(f"  [!] Lỗi kết nối trang: {e}")
                break

except KeyboardInterrupt:
    print("\n[!] ⛔ BẠN VỪA BẤM DỪNG (CTRL+C). Đang tiến hành lưu toàn bộ dữ liệu đã cào được...")

# 4. XUẤT DỮ LIỆU SẠCH (Đã bỏ cột Điện/Nước)
if all_rooms:
    df = pd.DataFrame(all_rooms)
    
    # Danh sách cột xuất file: BỎ Giá_Điện_VNĐ và Giá_Nước_VNĐ
    cols = ['Mã_Tin', 'Thời_Gian_Đăng', 'Thành_Phố', 'Quận_Huyện', 'Giá_Cho_Thuê', 'Diện_Tích_m2', 
            'Đầy_Đủ_Nội_Thất', 'Có_Gác_Lửng', 'Máy_Lạnh', 'Nóng_Lạnh', 'Thang_Máy', 
            'Tủ_Lạnh', 'Máy_Giặt', 'Bảo_Vệ_An_Ninh', 'Giờ_Tự_Do', 'Không_Chung_Chủ', 
            'Tiêu_Đề', 'Điện_Thoại', 'Địa_Chỉ_Chi_Tiết']
            
    df = df[[c for c in cols if c in df.columns]]
    df = df.dropna(subset=['Giá_Cho_Thuê', 'Diện_Tích_m2'])
    
    # Chống ghi trùng dòng (Lọc theo Mã_Tin)
    df = df.drop_duplicates(subset=['Mã_Tin'], keep='last')
    
    df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
    print(f"\n🎉 XUẤT SẮC! Đã lưu an toàn {len(df)} phòng trọ vào file '{FILE_NAME}'")
else:
    print("\n[!] Chưa cào được dữ liệu nào.")


🚀 BẮT ĐẦU CÀO DỮ LIỆU PHÒNG TRỌ (ĐÀ NẴNG, HÀ NỘI, TP.HCM)...
⚠️ LƯU Ý: Bạn có thể nhấn Ctrl + C bất cứ lúc nào để DỪNG và LƯU dữ liệu.

👉 ĐANG QUÉT: ĐÀ NẴNG (Bắt đầu từ Trang 1)
  + Trang 1 / 50...
    -> Đã lấy sơ bộ 19 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 19 phòng.
  + Trang 2 / 50...
    -> Đã lấy sơ bộ 20 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 3 / 50...
    -> Đã lấy sơ bộ 20 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 4 / 50...
    -> Đã lấy sơ bộ 20 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 5 / 50...
    -> Đã lấy sơ bộ 18 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 18 phòng.
  + Trang 6 / 50...
    -> Đã lấy sơ bộ 19 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 19 phòng.
  + Trang 7 / 50...
    -> Đã lấy sơ bộ 20 phòng. Đang vào chi tiết...
    -> Đã hoàn tất chi tiết 20 phòng.
  + Trang 8 / 50...
    -> Đã lấy sơ bộ 20 phòng. Đang vào